# Fase 2 — Agente com ferramentas e gravação confiável

**Problema desta fase:** sair de uma chain fixa para **um agente** (`create_agent`) que decide quais ferramentas usar (buscar cliente, consultar disponibilidade, ViaCEP, Minha Receita, propor visita/cadastro), sem nunca gravar sozinho um fato operacional.

**Conceitos da disciplina demonstrados:** ferramentas `@tool`, `create_agent`, prompt PTCF (Pessoa, Tarefa, Contexto, Formato) com few-shot, roteamento (`visita`/`cliente`/`geral`), callbacks de observabilidade.

**Decisões principais**
- **Propor ≠ confirmar.** O agente não tem ferramenta de confirmação; visita e cadastro PJ ficam pendentes até ação explícita do usuário (comando no terminal ou botão na aba Agenda do Streamlit). A confirmação é idempotente e auditada.
- Interpretações da fase 1 passam por uma **fila de revisão** (aceitar, corrigir, rejeitar, com versão); aceitar não cria projeto, pagamento nem compromisso.
- Projetos ficam ligados às mensagens-fonte por chave estrangeira.

**Evidências reais do agente (fora deste notebook):** [visita com revisão](../evidencias/fase2_visita_agente_revisao_manual.json), [CEP e idempotência](../evidencias/fase2_visita_cep_idempotencia_manual.json), [conflito de agenda](../evidencias/fase2_conflito_agente_manual.json), [CNPJ novo](../evidencias/fase2_cnpj_novo_agente_manual.json), [cadastro PJ com reteste](../evidencias/fase2_cadastro_pj_reteste_manual.json), [falha da busca local por CNPJ](../evidencias/fase2_falha_busca_local_cnpj_manual.json), [rota e callback](../evidencias/fase2_roteamento_callback_manual.json). Falhas e correções: [execucoes_e_falhas.md](../docs/execucoes_e_falhas.md); comparação de orquestradores: [fase2_orquestracao.md](../docs/fase2_orquestracao.md).

> **Fixtures:** nenhuma chamada à OpenAI, à Minha Receita ou ao ViaCEP acontece aqui. Respostas de API e eventos de callback são fixtures identificadas. A trajetória real do agente está nas evidências acima.

Notebooks de origem: `incrementos/03_revisao_persistente.ipynb`, `incrementos/04_cadastro_pj_agente.ipynb`, `incrementos/05_projetos_fontes.ipynb`, `incrementos/05b_orquestracao_agente.ipynb`.

---
*Origem desta parte: `incrementos/03_revisao_persistente.ipynb`.*

## Fase 2 — Início da revisão persistente
Este notebook usa as fixtures fictícias e os mesmos serviços do terminal, sem API/LLM.
Aceitar confirma a interpretação; não cria projeto, pagamento ou compromisso.
Demonstra autoria, correção, fontes imutáveis, versão e persistência. Agente/ferramentas/APIs ainda serão implementados.

In [1]:
# CONTEXTO: identidade de demonstração é entrada do serviço, não um nome extraído pelo modelo.
from pathlib import Path
import sys, tempfile
raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/secretaria').is_dir())
sys.path.insert(0,str(raiz/'src'))
from secretaria.banco import Banco,Contexto
from secretaria.extracao import analisar_demonstracao
from secretaria.fixtures_extracao import criar_modelo_fixture
from secretaria.fila_revisao import FilaRevisao
temporario=tempfile.TemporaryDirectory()
caminho=Path(temporario.name)/'revisao.sqlite3'
banco=Banco(caminho);banco.preparar_demo()
ctx=Contexto('demo','ana')
print('Demonstração local: banco temporário, sem API.')

Demonstração local: banco temporário, sem API.


In [2]:
# CHAIN/FILA: extração manual identificada produz propostas; fila revalida suas fontes.
resultado=analisar_demonstracao(banco,ctx,criar_modelo_fixture(),'fixture_manual')
fila=FilaRevisao(banco)
cadastro=fila.registrar_resultado(ctx,resultado)
assert cadastro['novas']==8
assert fila.registrar_resultado(ctx,resultado)['novas']==0
propostas=fila.listar(ctx)
print('Interpretações pendentes:',len(propostas))

Interpretações pendentes: 8


In [3]:
# REVISÃO: corrige interpretação, mantém fontes e confirma a versão efetivamente exibida.
primeira=propostas[0]
corrigida=fila.corrigir(ctx,primeira['id'],primeira['versao'],
    {'resumo':'Implantação do portal de atendimento da Alfa.'},'Resumo conferido neste exercício.')
assert corrigida['dados']['fontes']==primeira['dados']['fontes']
aceita=fila.decidir(ctx,corrigida['id'],corrigida['versao'],'aceitar','Conferi a interpretação e as fontes.')
assert aceita['estado']=='aceita'
assert fila.decidir(ctx,aceita['id'],corrigida['versao'],'aceitar','Repetição')==aceita
segunda=propostas[1]
rejeitada=fila.decidir(ctx,segunda['id'],segunda['versao'],'rejeitar','Exemplo didático de rejeição humana.')
print('Decisões:',aceita['estado'],rejeitada['estado'])

Decisões: aceita rejeitada


In [4]:
# VERSÃO: uma tela antiga não pode confirmar conteúdo corrigido por outra operação.
terceira=propostas[2]
fila.corrigir(ctx,terceira['id'],terceira['versao'],{'resumo':'Entrada prevista, sem comprovação de pagamento.'},'Esclarecer previsão.')
try:
    fila.decidir(ctx,terceira['id'],terceira['versao'],'aceitar','Tela antiga')
    raise AssertionError('Deveria bloquear a versão antiga.')
except ValueError as erro:
    print('Bloqueio esperado:',erro)
try:
    fila.obter(Contexto('demo','bruno'),primeira['id'])
    raise AssertionError('Deveria bloquear outro usuário.')
except PermissionError:
    print('Fila pessoal de outro usuário bloqueada.')

Bloqueio esperado: Versão mudou. Leia a proposta atual antes de decidir.
Fila pessoal de outro usuário bloqueada.


In [5]:
# PERSISTÊNCIA/AUDITORIA: reabre SQLite e confere decisões e antes/depois.
banco.fechar()
banco=Banco(caminho);fila=FilaRevisao(banco)
assert fila.obter(ctx,primeira['id'])['estado']=='aceita'
assert fila.obter(ctx,segunda['id'])['estado']=='rejeitada'
historico=fila.historico(ctx,primeira['id'])
assert [h['acao'] for h in historico]==['registrar','corrigir','aceitar']
assert fila.registrar_resultado(ctx,resultado)['novas']==0
# As tabelas operacionais existem desde as migrações das fases seguintes; aceitar não pode preenchê-las.
operacionais={t:banco.con.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
              for t in ('projetos','orcamentos','pagamentos','agendamentos')}
assert set(operacionais.values())=={0}, operacionais
print('Histórico:',[h['acao'] for h in historico])
print('Decisões persistiram; nenhum registro operacional foi criado.')

Histórico: ['registrar', 'corrigir', 'aceitar']
Decisões persistiram; nenhum registro operacional foi criado.


In [6]:
# LIMPEZA: os dados deste notebook são temporários; não modifica o banco de demonstração do terminal.
banco.fechar()
temporario.cleanup()
print('Notebook concluído sem API e sem efeitos operacionais.')

Notebook concluído sem API e sem efeitos operacionais.


---
*Origem desta parte: `incrementos/04_cadastro_pj_agente.ipynb`.*

## Cadastro PJ por proposta e confirmação

Demonstração local da fase 2. A consulta abaixo é uma **fixture manual**: não chama Minha Receita nem LLM. O agente real usa as ferramentas `consultar_cnpj_minha_receita` e `propor_cadastro_cliente`; a confirmação fica no comando `/confirmar-cliente ID`.


In [7]:
import sys, tempfile
from pathlib import Path

projeto = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/secretaria').is_dir())
sys.path.insert(0, str(projeto / 'src'))
from secretaria.banco import Banco, Contexto
from secretaria.agenda import Agenda, agora_utc

temporario = tempfile.TemporaryDirectory()
caminho = Path(temporario.name) / 'secretaria.sqlite3'
banco = Banco(caminho)
banco.preparar_demo()
ctx = Contexto('demo', 'ana')
agenda = Agenda(banco)
print('Versão SQLite:', banco.con.execute('PRAGMA user_version').fetchone()[0])


Versão SQLite: 11


In [8]:
fixture = {'origem': 'fixture_local', 'cnpj': '33683111000280',
           'razao_social': 'Empresa Exemplo — fixture', 'nome_fantasia': 'Exemplo',
           'consultado_em': agora_utc().isoformat()}
consulta_id = agenda.registrar_consulta_cnpj(ctx, fixture)
proposta = agenda.propor_cadastro_cliente(ctx, consulta_id)
assert proposta['estado'] == 'pendente'
assert banco.con.execute('SELECT COUNT(*) FROM clientes').fetchone()[0] == 0
print(proposta)
print('Clientes antes da confirmação: 0')


{'proposta_id': 1, 'estado': 'pendente', 'cnpj': '33683111000280', 'razao_social': 'Empresa Exemplo — fixture', 'nome_fantasia': 'Exemplo', 'origem': 'fixture_local', 'aviso': 'Cliente ainda não cadastrado. Para gravar, o usuário precisa confirmar a proposta 1 explicitamente.'}
Clientes antes da confirmação: 0


In [9]:
confirmacao = agenda.confirmar_cadastro_cliente(ctx, proposta['proposta_id'])
repeticao = agenda.confirmar_cadastro_cliente(ctx, proposta['proposta_id'])
assert confirmacao['cliente_id'] == repeticao['cliente_id']
assert repeticao['estado'] == 'ja_confirmada'
banco.fechar()
with Banco(caminho) as reaberto:
    cliente = reaberto.con.execute('SELECT nome,cnpj,origem_cadastro FROM clientes WHERE id=?',
                                   (confirmacao['cliente_id'],)).fetchone()
    auditoria = reaberto.con.execute('SELECT COUNT(*) FROM auditoria_clientes').fetchone()[0]
    assert cliente['origem_cadastro'] == 'fixture_local' and auditoria == 1
    print('Cliente após reabrir:', dict(cliente), '| auditorias:', auditoria)
temporario.cleanup()


Cliente após reabrir: {'nome': 'Empresa Exemplo — fixture', 'cnpj': '33683111000280', 'origem_cadastro': 'fixture_local'} | auditorias: 1


---
*Origem desta parte: `incrementos/05_projetos_fontes.ipynb`.*

## Projeto e mensagens-fonte — fase 2

Demonstração com **conversa fictícia** e **fixture local**. Aceitar uma interpretação na fila não grava o projeto. Uma segunda confirmação cria o projeto e preserva vínculos reais com as mensagens no SQLite. A anotação financeira abaixo documenta o que foi dito; **não é pagamento**.


In [10]:
import sys, tempfile
from pathlib import Path

projeto = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/secretaria').is_dir())
sys.path.insert(0, str(projeto/'src'))
from secretaria.banco import Banco, Contexto
from secretaria.extracao import analisar_snapshot
from secretaria.fixtures_extracao import criar_modelo_fixture
from secretaria.fila_revisao import FilaRevisao
from secretaria.projetos_fontes import ProjetosFontes

temporario=tempfile.TemporaryDirectory()
caminho=Path(temporario.name)/'secretaria.sqlite3'
banco=Banco(caminho)
banco.preparar_demo()
ctx=Contexto('demo','ana')
corpus=banco.selecionar_corpus(ctx,'Demonstração fictícia')
banco.importar(ctx,corpus,projeto/'dados/ficticios/conversa_empresa.txt')
print('Versão SQLite:',banco.con.execute('PRAGMA user_version').fetchone()[0])


Versão SQLite: 11


In [11]:
resultado=analisar_snapshot(banco.snapshot_analise(ctx,corpus),criar_modelo_fixture(),'fixture_manual')
fila=FilaRevisao(banco)
ids=fila.registrar_resultado(ctx,resultado)['ids']
id_projeto,id_financeiro=ids[0],ids[1]
fila.decidir(ctx,id_projeto,1,'aceitar','Fonte do projeto conferida.')
fila.decidir(ctx,id_financeiro,1,'aceitar','Orçamento informado; não é pagamento.')
assert banco.con.execute('SELECT COUNT(*) FROM projetos').fetchone()[0]==0
print('Projetos após aceitar interpretações: 0')


Projetos após aceitar interpretações: 0


In [12]:
servico=ProjetosFontes(banco)
previa=servico.prever(ctx,id_projeto)
print('Prévia:',previa)
criacao=servico.confirmar(ctx,id_projeto,previa['versao'])
assert criacao['estado']=='confirmada'
print('Projeto confirmado:',criacao)
print('Fontes:',servico.fontes(ctx,criacao['anotacao_id']))


Prévia: {'proposta_id': 1, 'versao': 2, 'acao': 'criar_projeto', 'projeto': 'Alfa', 'projeto_id': None, 'cliente_id': None, 'area': 'projeto', 'natureza': 'relato', 'resumo': 'Implantação do portal; responsável informado Ana.', 'fontes': [{'mensagem_id': 3, 'trecho': 'Projeto Alfa: implantação do portal de atendimento.\nCliente: Alfa Serviços.\nResponsável: Ana.'}], 'limite': 'Anotação financeira não é pagamento; anotação de agenda não reserva horário.'}
Projeto confirmado: {'estado': 'confirmada', 'projeto_id': 1, 'anotacao_id': 1, 'fontes_vinculadas': 1}
Fontes: [{'mensagem_id': 3, 'trecho_snapshot': 'Projeto Alfa: implantação do portal de atendimento.\nCliente: Alfa Serviços.\nResponsável: Ana.', 'data_original': '09/11/2026, 08:02:00', 'autor': 'Ana'}]


In [13]:
previa_financeira=servico.prever(ctx,id_financeiro,projeto_id=criacao['projeto_id'])
anotacao=servico.confirmar(ctx,id_financeiro,previa_financeira['versao'],projeto_id=criacao['projeto_id'])
repeticao=servico.confirmar(ctx,id_financeiro,previa_financeira['versao'],projeto_id=criacao['projeto_id'])
assert repeticao['estado']=='ja_confirmada'
banco.fechar()
with Banco(caminho) as reaberto:
    fontes=ProjetosFontes(reaberto).fontes(ctx,anotacao['anotacao_id'])
    assert fontes
    print('Fontes após reabrir:',fontes)
    print('Anotações:',reaberto.con.execute('SELECT COUNT(*) FROM anotacoes_projeto').fetchone()[0])
temporario.cleanup()


Fontes após reabrir: [{'mensagem_id': 4, 'trecho_snapshot': 'Orçamento Alfa: R$ 10.000,00. Entrada combinada de 30%; restante em 20/11/2026.', 'data_original': '09/11/2026, 08:03:00', 'autor': 'Bruno'}]
Anotações: 2


---
*Origem desta parte: `incrementos/05b_orquestracao_agente.ipynb`.*

## Fase 2 — prompt, rota e callbacks

Demonstração local dos componentes usados pelo mesmo agente do menu 9. As entradas são **fixtures locais**; nenhuma chamada à OpenAI ou às APIs de negócio ocorre neste notebook. O teste real do agente é feito no terminal.


In [14]:
from pathlib import Path
import sys
raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/secretaria').is_dir())
sys.path.insert(0, str(raiz / 'src'))
from secretaria.agente_agenda import montar_prompt_agenda
from secretaria.roteamento import roteador_pedido
from secretaria.observabilidade import RastroAgente


### Prompt PTCF e exemplos
A estrutura delimita papel, tarefa, regras e formato. Os exemplos são instruções de comportamento; não são respostas medidas de um modelo.


In [15]:
prompt = montar_prompt_agenda('ana')
assert all(marca in prompt for marca in ('[PESSOA]', '[TAREFA]', '[CONTEXTO E REGRAS]', '[EXEMPLOS]', '[FORMATO]'))
print('Seções:', [linha for linha in prompt.splitlines() if linha.startswith('[')])


Seções: ['[PESSOA]', '[TAREFA]', '[CONTEXTO E REGRAS]', '[EXEMPLOS]', '[FORMATO]']


### Roteamento dinâmico leve
A rota orienta o agente; não confirma visitas nem restringe ferramentas. Os serviços Python continuam validando efeitos e identidade.


In [16]:
casos = ['Agende uma visita ao cliente Alfa', 'O CNPJ está cadastrado?', 'Olá']
rotas = [roteador_pedido.invoke({'pergunta': pergunta, 'anteriores': []})['rota'] for pergunta in casos]
assert rotas == ['visita', 'cliente', 'geral']
print(list(zip(casos, rotas)))


[('Agende uma visita ao cliente Alfa', 'visita'), ('O CNPJ está cadastrado?', 'cliente'), ('Olá', 'geral')]


### Callback sem conteúdo privado
Eventos abaixo são acionados **manualmente** para verificar o formato do rastro. Eles não demonstram uma chamada real ao agente.


In [17]:
from uuid import uuid4
rastro = RastroAgente()
id_execucao = uuid4()
rastro.on_chat_model_start({}, [], run_id=id_execucao)
rastro.on_tool_start({'name': 'buscar_cliente'}, 'entrada omitida', run_id=id_execucao)
rastro.on_tool_end('retorno omitido', run_id=id_execucao)
rastro.on_llm_end(None, run_id=id_execucao)
assert [e['evento'] for e in rastro.eventos] == ['modelo_iniciado', 'ferramenta_iniciada', 'ferramenta_concluida', 'modelo_concluido']
print(rastro.eventos)


[{'evento': 'modelo_iniciado', 'run_id': '11526911-d9d1-4868-ae01-db76ef8f9068', 'instante_utc': '2026-11-12T01:54:15.202824+00:00'}, {'evento': 'ferramenta_iniciada', 'run_id': '11526911-d9d1-4868-ae01-db76ef8f9068', 'instante_utc': '2026-11-12T01:54:15.202824+00:00', 'nome': 'buscar_cliente'}, {'evento': 'ferramenta_concluida', 'run_id': '11526911-d9d1-4868-ae01-db76ef8f9068', 'instante_utc': '2026-11-12T01:54:15.202824+00:00'}, {'evento': 'modelo_concluido', 'run_id': '11526911-d9d1-4868-ae01-db76ef8f9068', 'instante_utc': '2026-11-12T01:54:15.202824+00:00'}]
